In [ ]:
using Plots
using Random
using StaticArrays
using Images

include("simulation.jl")
include("src/plotting.jl")
include("src/animations.jl")


In [ ]:
# Set up simulation parameters
mid_landmarks = [
    @SVector[165.0, 50.0, 0.0], 
]
Random.seed!(1337)

max_cost = 5.0

vo_params = VO.VOParams(
            fovRadius=75.0, 
            fovAngle=90*π/180, 
            errorRate=0.03, 
            updateRate=5.0, 
            minFeatures=6,
            maxError=max_cost
        )

unsafe_zone = World.Circle(@SVector[150.0, 150.0], 30.0)

setup = setup_simulation(
    image_path="images/field.png",
    domain_size=200.0,
    seed=1337,
    goal_pos=[230.0, 175.0, 1.2*π/2],
    start_pos=[30.0, 130.0, -π/2],
    mid_landmarks=mid_landmarks,
    unsafe_zones=[unsafe_zone],
    max_cost=max_cost,
    vo_params=vo_params,
    nominal_iterations=10000,
    backup_iterations=100
)

# Extract key variables for convenience
features = setup.features
AR = setup.AR
domain_size = setup.domain_size
nominal_planner = setup.nominal_planner
backup_planner = setup.backup_planner
vo_params = setup.vo_params
x0 = setup.x0
goal = setup.goal
max_cost = setup.max_cost

println("Setup complete!")


In [ ]:
using Images
image = load("images/field.png")
plot([0,domain_size*AR],[0,domain_size],reverse(image, dims = 1), yflip = false, aspect_ratio = :auto)
xlims!(0, domain_size*AR)
ylims!(0, domain_size)
plotFeatures2D(features, :red)
plot!(unsafe_zone)
plot!(legend = false, grid = false, axis = false)

In [ ]:
# Run the main simulation

sim_results = @time run_simulation(setup, max_iterations=800, verbose=true, use_controller=true)

# Extract results for convenience
x_hist = sim_results.x_hist
x_est_hist = sim_results.x_est_hist
gk_hist = sim_results.gk_hist
features_hist = sim_results.features_hist
backup_hist = sim_results.backup_hist
cost_hist = sim_results.cost_hist
estimation_error_hist = sim_results.estimation_error_hist
num_feats_hist = sim_results.num_feats_hist
new_committed_hist = sim_results.new_committed_hist
gk_times = sim_results.gk_times
reroot_times = sim_results.reroot_times
t_f = sim_results.t_f
ΔT = sim_results.ΔT
control_hist = sim_results.control_hist
landmarks_discovered_hist = sim_results.landmarks_discovered_hist

println("Simulation complete!")
println("Final time: $t_f seconds")

In [ ]:
# Compute timing statistics
timing_stats = compute_timing_stats(sim_results)
println("Reroot time: $(timing_stats.mean_reroot) ± $(timing_stats.std_reroot) ms")
println("Gatekeeper time: $(timing_stats.mean_gk) ± $(timing_stats.std_gk) ms")
plot_timing_histograms(sim_results)
savefig("timing_histograms.svg")

In [ ]:
default(dpi=600, size=(600,400))
anim = animate_simulation(x_hist, x_est_hist, gk_hist, features_hist, backup_hist,
                          nominal_planner, backup_planner, vo_params, domain_size, AR,
                          cost_hist=cost_hist, num_feats_hist=num_feats_hist, 
                          landmarks_discovered_hist=landmarks_discovered_hist,
                          frame_skip=10, show_estimate=true, show_unseen = false,
                          frame_ext="png", output_dir="frames",
                          show_text=true,show_labels=false,
                        #   save_frames=[1 70 250 430], 
                          save_frames=nothing
                          )

In [ ]:
# gif(anim, "field.gif", fps = 10)
default(dpi=600, size=(600,400))
mp4(anim, "animations/field_test_5m.mp4", fps = 10)

In [ ]:
plot()

using Images
image = load("images/field.png")
plot!([0,domain_size*AR],[0,domain_size],reverse(image, dims = 1), yflip = false, aspect_ratio = :auto)
x_hist_x = [x[1] for x in x_hist]
x_hist_y = [x[2] for x in x_hist]
# push!(x_hist_x, goal[1])
# push!(x_hist_y, goal[2])
plotFeatures2D(features_hist[end], :white)
plot!(x_hist_x, x_hist_y, color = :blue, linewidth = 2)
for  l in backup_planner.prob.landmarks
      plot!([l[1]], [l[2]], color = :cyan, marker = :circle)
      plot!(Shape(l[1] .+ backup_planner.prob.turning_radius*cos.(0:0.01:2pi),
                  l[2] .+ backup_planner.prob.turning_radius*sin.(0:0.01:2pi)), 
            color=:cyan, alpha=0.2)
end
plot!(unsafe_zone)
plot!([x_hist[1][1]], [x_hist[1][2]], color = :green, marker = :circle)
plot!(legend=false)
# plot!([mid_landmark[1]], [mid_landmark[2]], color = :yellow, marker = :circle, legend  = false)

In [ ]:
plot_font = "sans-serif"
default(
    fontfamily=plot_font,
    linewidth=2,
    guidefontsize=16,
    tickfontsize=16,
    legendfontsize=10,
    titlefontsize=18
)
using Measures


In [ ]:
plot((1:length(cost_hist))*ΔT, cost_hist, xlabel="Time [s]", ylabel="Pos. Error [m]", legend=false)
plot!([(1, max_cost), (length(cost_hist)*ΔT, max_cost)], linestyle=:dash, color=:red)
plot!(size=(600,400))
# plot!((1:length(estimation_error_hist))*ΔT, estimation_error_hist, color=:green)
savefig("position_error_plot2.svg")

In [ ]:
plot()
plot!((1:length(num_feats_hist))*ΔT, num_feats_hist, xlabel="Time [s]", ylabel="# Features Seen", legend=false)
plot!([(1, vo_params.minFeatures), (length(num_feats_hist)*ΔT, vo_params.minFeatures)], linestyle=:dash, color=:red)
x_start = NaN
for i in 2:length(num_feats_hist)
    if !isnan(num_feats_hist[i-1]) && isnan(num_feats_hist[i])
        x_start = (i-1) * ΔT
    elseif isnan(num_feats_hist[i-1]) && !isnan(num_feats_hist[i]) && !isnan(x_start)
        x_end = (i-1) * ΔT
        plot!(Shape([x_start, x_end, x_end, x_start], [0, 0, 10000, 10000]), color=:gray, alpha=0.3, legend=false)
        x_start = NaN
    end
end
ylims!(0, 120)
plot!(size=(600,400),bottom_margin=7mm, left_margin=5mm)
savefig("num_features_plot2.svg")

In [ ]:
dist_to_goal = [norm(x[SOneTo(2)] - backup_planner.prob.landmarks[end]) for x in x_hist]
plot((1:length(dist_to_goal))*ΔT, dist_to_goal, xlabel="Time", ylabel="Distance to goal", legend=false)

In [ ]:
# Set up omniscient planner for comparison
omniscient = @time setup_omniscient_planner(setup, iterations=5000)

omniscient_cost, omniscient_path = compute_omniscient_cost(omniscient, x0)
println("omniscient cost: ", omniscient_cost)


In [ ]:
# Compute nominal trajectory failure point
_, initial_nom_path = query_nominal_planner(nominal_planner, x0)
initial_nom_path = make_path_from_waypoints(initial_nom_path, nominal_planner.prob.turning_radius)

nom_cost_total, x_nom_fail, nom_fail_idx = compute_nominal_failure(nominal_planner, backup_planner.prob, initial_nom_path, vo_params)
println("nominal cost: ", nom_cost_total, " failed at ", x_nom_fail)

# Plot comparison
image = load("images/field.png")
plot([0,domain_size*AR],[0,domain_size],reverse(image, dims = 1), yflip = false, aspect_ratio = :auto)
xlims!(0, domain_size*AR)
ylims!(0, domain_size)
plot!(Shape(backup_planner.prob.landmarks[end][1] .+ backup_planner.prob.turning_radius*cos.(0:0.01:2pi),
            backup_planner.prob.landmarks[end][2] .+ backup_planner.prob.turning_radius*sin.(0:0.01:2pi)), 
      color=:cyan, alpha=0.2)

for p in omniscient_path
    plot!(p, linewidth=5, color=:magenta)
end
for p in initial_nom_path
    plot!(p, linewidth=5, color=:black)
end
plotFOV(x_nom_fail[1:2], x_nom_fail[3], vo_params.fovRadius, vo_params.fovAngle)
plot!(nominal_planner.prob.unsafe_zones; color=:red, width=2)
plot!([x0[1]], [x0[2]], color = :orange, marker = :circle, markersize = 10)
plot!([backup_planner.prob.landmarks[end][1]], [backup_planner.prob.landmarks[end][2]], color = :yellow, marker = :circle, markersize = 10)
plot!(unsafe_zone; color=:red, width=2)
plotFeatures2D(features, :white)

xlabel!("East [m]")
ylabel!("North [m]")
plot!(legend=false, grid=false)
plot!(size=(800,600))
savefig("planner_comparison.svg")